### SCD Type 1 on Branch Table


In [0]:
from pyspark.sql.functions import *
#  Source data
df = spark.read.format("delta").table("bankingdata.silver.branch")

In [0]:
df.columns

### Hash Column at Source

In [0]:
#  Add HASH column 

df_hash = df.withColumn(
    "src_hash",
    crc32(concat_ws("||",
        col("BranchID").cast("string"),
        col("BranchName"),
        col("City"),
        col("Province"),
        col("ModifiedDate").cast("string")
    ))
)

### Target Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bankingdata.gold.dim_branch
(
    BranchID INT,
    BranchName STRING,
    City STRING,
    Province STRING,
    ModifiedDate TIMESTAMP,
    
    HASHVALUE STRING,
    CREATEDDATE TIMESTAMP,
    CREATEDBY STRING,
    UPDATEDDATE TIMESTAMP,
    UPDATEDBY STRING
)

In [0]:
# Target table
from delta.tables import DeltaTable
table_name = "bankingdata.gold.dim_branch"
delta_tgt = DeltaTable.forName(spark, table_name)


### SCD 1 Merge Logic


In [0]:
# MERGE (SCD TYPE 1)
(
    delta_tgt.alias("tgt")
    .merge(
        df_hash.alias("src"),
        "tgt.BranchID = src.BranchID"
    )
    .whenMatchedUpdate(
        condition="tgt.HASHVALUE != src.src_hash",
        set={
            "BranchID": "src.BranchID",
            "BranchName": "src.BranchName",
            "City": "src.City",
            "Province": "src.Province",
            "ModifiedDate": "src.ModifiedDate",
            "HASHVALUE": "src.src_hash",
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("databricks-updated")
        }
    )
    .whenNotMatchedInsert(
        values={
            "BranchID": "src.BranchID",
            "BranchName": "src.BranchName",
            "City": "src.City",
            "Province": "src.Province",
            "ModifiedDate": "src.ModifiedDate",
            "HASHVALUE": "src.src_hash",
            "CREATEDDATE": current_timestamp(),
            "CREATEDBY": lit("databricks"),
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("databricks")
        }
    )
    .execute()
)

In [0]:
%sql
select * from bankingdata.gold.dim_branch